Data Prep

In [ ]:
# ============================================================
# CELL 1: DATA PREP
# ============================================================
import os
import zipfile
import pandas as pd
from pathlib import Path
from sklearn.model_selection import KFold

# ---- 1. Unzip ----
ZIP_PATH = "/content/feedback-prize-2021.zip"   # change to your zip path
EXTRACT_DIR = Path("./data")

if not EXTRACT_DIR.exists():
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(EXTRACT_DIR)
    print(f"Extracted to {EXTRACT_DIR}")
else:
    print(f"{EXTRACT_DIR} already exists, skipping unzip")

# Adjust these if your zip structure is different
TRAIN_DIR = EXTRACT_DIR / "train"
TEST_DIR = EXTRACT_DIR / "test"
TRAIN_CSV = EXTRACT_DIR / "train.csv"

# ---- 2. Load annotations ----
train_data = pd.read_csv(TRAIN_CSV, dtype={"discourse_id": str})
train_data["id"] = train_data["id"].astype(str)
train_data["discourse_start"] = train_data["discourse_start"].astype(int)
train_data["discourse_end"] = train_data["discourse_end"].astype(int)
print(f"Annotations: {train_data.shape}")
print(f"Discourse types: {train_data['discourse_type'].unique()}")

# ---- 3. Load essays ----
def load_essays(folder):
    rows = []
    for fp in Path(folder).glob("*.txt"):
        with open(fp, "r", encoding="utf-8") as f:
            rows.append({"id": fp.stem, "text": f.read()})
    return pd.DataFrame(rows)

ready_df = load_essays(TRAIN_DIR)
test_df = load_essays(TEST_DIR)
ready_df["id"] = ready_df["id"].astype(str)
test_df["id"] = test_df["id"].astype(str)
print(f"Train essays: {len(ready_df)} | Test essays: {len(test_df)}")

# ---- 4. Sanity checks ----
missing = set(train_data["id"]) - set(ready_df["id"])
print(f"Annotated essays missing from train folder: {len(missing)}")

# Confirm spans align with text
sample = train_data.iloc[0]
essay_text = ready_df.loc[ready_df["id"] == sample["id"], "text"].iloc[0]
sliced = essay_text[sample["discourse_start"]:sample["discourse_end"]]
print("\n--- Span alignment check ---")
print("CSV:    ", sample["discourse_text"][:80])
print("Sliced: ", sliced[:80])
print("Match:  ", sliced.strip() == sample["discourse_text"].strip())

# Length distribution
ready_df["n_words"] = ready_df["text"].str.split().str.len()
print(f"\nWord counts — median: {ready_df['n_words'].median():.0f}, "
      f"95th: {ready_df['n_words'].quantile(0.95):.0f}, "
      f"max: {ready_df['n_words'].max()}")

# ---- 5. K-fold split ----
kf = KFold(n_splits=5, shuffle=True, random_state=42)
ready_df["kfold"] = -1
for f, (_, val_idx) in enumerate(kf.split(ready_df)):
    ready_df.loc[val_idx, "kfold"] = f
print(f"\nFold sizes:\n{ready_df['kfold'].value_counts().sort_index()}")

# ---- 6. Save processed DFs (optional, speeds up reruns) ----
ready_df.to_pickle("ready_df.pkl")
test_df.to_pickle("test_df.pkl")
train_data.to_pickle("train_data.pkl")
print("\nSaved: ready_df.pkl, test_df.pkl, train_data.pkl")
print(f"\nReady: ready_df={ready_df.shape}, train_data={train_data.shape}, test_df={test_df.shape}")

Training

In [ ]:
# ============================================================
# CELL 2: TRAINING
# ============================================================
import os, gc, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import LongformerTokenizerFast, LongformerModel, LongformerConfig, get_cosine_schedule_with_warmup
from tqdm import tqdm

# ---- Config ----
class CFG:
    model_name = "allenai/longformer-base-4096"
    max_length = 1024
    train_batch_size = 4
    valid_batch_size = 4
    epochs = 5
    lr = 3e-5
    weight_decay = 0.01
    warmup_ratio = 0.1
    seed = 42
    fold = 0                 # which fold to train; loop over folds for full CV
    output_dir = "./output"

os.makedirs(CFG.output_dir, exist_ok=True)

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
seed_everything(CFG.seed)

# ---- Labels (BIO scheme) ----
DISCOURSE_TYPES = ["Lead", "Position", "Claim", "Counterclaim", "Rebuttal", "Evidence", "Concluding Statement"]
LABELS = ["O"] + [f"{p}-{t}" for t in DISCOURSE_TYPES for p in ["B", "I"]]
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for i, l in enumerate(LABELS)}
NUM_LABELS = len(LABELS)
print(f"Num labels: {NUM_LABELS}")

# ---- Dataset ----
class FeedbackDataset(Dataset):
    def __init__(self, df, tokenizer, max_length, train_data=None, is_train=True):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.is_train = is_train
        self.anno = train_data.groupby("id") if train_data is not None else None

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        essay_id, text = row["id"], row["text"]

        enc = self.tokenizer(
            text, max_length=self.max_length, padding="max_length",
            truncation=True, return_offsets_mapping=True, return_tensors="pt",
        )
        input_ids = enc["input_ids"].squeeze(0)
        attention_mask = enc["attention_mask"].squeeze(0)
        offsets = enc["offset_mapping"].squeeze(0).numpy()

        global_attention_mask = torch.zeros_like(input_ids)
        global_attention_mask[0] = 1   # CLS gets global attention

        labels = np.full(self.max_length, -100, dtype=np.int64)

        if self.is_train and self.anno is not None:
            for i, (s, e) in enumerate(offsets):
                if s == 0 and e == 0:
                    labels[i] = -100
                else:
                    labels[i] = label2id["O"]

            try:
                ann = self.anno.get_group(essay_id)
            except KeyError:
                ann = pd.DataFrame()

            for _, a in ann.iterrows():
                d_start, d_end, d_type = int(a["discourse_start"]), int(a["discourse_end"]), a["discourse_type"]
                first = True
                for i, (s, e) in enumerate(offsets):
                    if s == 0 and e == 0:
                        continue
                    if s >= d_start and e <= d_end:
                        labels[i] = label2id[f"B-{d_type}" if first else f"I-{d_type}"]
                        first = False

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "global_attention_mask": global_attention_mask,
            "labels": torch.tensor(labels, dtype=torch.long),
            "offset_mapping": torch.tensor(offsets, dtype=torch.long),
            "essay_id": essay_id,
        }

# ---- Model ----
class FeedbackModel(nn.Module):
    def __init__(self, model_name, num_labels):
        super().__init__()
        self.config = LongformerConfig.from_pretrained(model_name)
        self.backbone = LongformerModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask, global_attention_mask, labels=None):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask,
                            global_attention_mask=global_attention_mask)
        logits = self.classifier(self.dropout(out.last_hidden_state))
        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss(ignore_index=-100)(logits.view(-1, logits.size(-1)), labels.view(-1))
        return {"loss": loss, "logits": logits}

# ---- Train ----
def train_one_fold(ready_df, train_data, fold):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    train_split = ready_df[ready_df["kfold"] != fold].reset_index(drop=True)
    valid_split = ready_df[ready_df["kfold"] == fold].reset_index(drop=True)
    print(f"Train: {len(train_split)} | Valid: {len(valid_split)}")

    tokenizer = LongformerTokenizerFast.from_pretrained(CFG.model_name, add_prefix_space=True)
    train_ds = FeedbackDataset(train_split, tokenizer, CFG.max_length, train_data, is_train=True)
    valid_ds = FeedbackDataset(valid_split, tokenizer, CFG.max_length, train_data, is_train=True)
    train_loader = DataLoader(train_ds, batch_size=CFG.train_batch_size, shuffle=True, num_workers=2, pin_memory=True)
    valid_loader = DataLoader(valid_ds, batch_size=CFG.valid_batch_size, shuffle=False, num_workers=2, pin_memory=True)

    model = FeedbackModel(CFG.model_name, NUM_LABELS).to(device)
    optimizer = AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
    total_steps = len(train_loader) * CFG.epochs
    scheduler = get_cosine_schedule_with_warmup(
        optimizer, int(total_steps * CFG.warmup_ratio), total_steps
    )
    scaler = torch.cuda.amp.GradScaler()
    best_val = float("inf")

    for epoch in range(CFG.epochs):
        model.train()
        running, n = 0.0, 0
        pbar = tqdm(train_loader, desc=f"Fold {fold} Epoch {epoch+1}/{CFG.epochs}")
        for batch in pbar:
            optimizer.zero_grad()
            with torch.cuda.amp.autocast():
                out = model(
                    batch["input_ids"].to(device),
                    batch["attention_mask"].to(device),
                    batch["global_attention_mask"].to(device),
                    batch["labels"].to(device),
                )
                loss = out["loss"]
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update(); scheduler.step()
            running += loss.item(); n += 1
            pbar.set_postfix(loss=f"{running/n:.4f}")

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in tqdm(valid_loader, desc="Validating"):
                with torch.cuda.amp.autocast():
                    out = model(
                        batch["input_ids"].to(device),
                        batch["attention_mask"].to(device),
                        batch["global_attention_mask"].to(device),
                        batch["labels"].to(device),
                    )
                val_loss += out["loss"].item()
        val_loss /= len(valid_loader)
        print(f"Epoch {epoch+1} | val_loss = {val_loss:.4f}")

        if val_loss < best_val:
            best_val = val_loss
            torch.save(model.state_dict(), f"{CFG.output_dir}/longformer_fold{fold}.pt")
            print(f"  ✓ saved (val_loss={val_loss:.4f})")

    del model, optimizer, scheduler
    gc.collect(); torch.cuda.empty_cache()
    return best_val

# Load prepped data and train
ready_df = pd.read_pickle("ready_df.pkl")
train_data = pd.read_pickle("train_data.pkl")

best = train_one_fold(ready_df, train_data, fold=CFG.fold)
print(f"\nFold {CFG.fold} best val_loss: {best:.4f}")

In [ ]:
# ============================================================
# CELL 3: INFERENCE + SAMPLE RESULTS
# ============================================================
import re
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from transformers import LongformerTokenizerFast
from tqdm import tqdm

# Which fold(s) to use for inference. Add more paths to ensemble.
VAL_FOLD = 0
FOLDS_TO_USE = [0]  # e.g. [0, 1, 2] to ensemble

# Tunable post-processing thresholds
MIN_WORDS = {"Lead": 9, "Position": 5, "Claim": 3, "Counterclaim": 6,
             "Rebuttal": 4, "Evidence": 14, "Concluding Statement": 11}

def predict(test_df, model_paths, tokenizer):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    test_ds = FeedbackDataset(test_df, tokenizer, CFG.max_length, train_data=None, is_train=False)
    test_loader = DataLoader(test_ds, batch_size=CFG.valid_batch_size, shuffle=False, num_workers=2)

    all_probs = None
    for path in model_paths:
        model = FeedbackModel(CFG.model_name, NUM_LABELS).to(device)
        model.load_state_dict(torch.load(path, map_location=device))
        model.eval()
        fold_probs = []
        with torch.no_grad():
            for batch in tqdm(test_loader, desc=f"Predict {path}"):
                with torch.amp.autocast("cuda"):
                    out = model(
                        batch["input_ids"].to(device),
                        batch["attention_mask"].to(device),
                        batch["global_attention_mask"].to(device),
                    )
                fold_probs.append(torch.softmax(out["logits"], -1).cpu().numpy())
        fold_probs = np.concatenate(fold_probs, 0)
        all_probs = fold_probs if all_probs is None else all_probs + fold_probs
        del model; torch.cuda.empty_cache()
    all_probs /= len(model_paths)

    submissions = []
    for idx, row in test_df.reset_index(drop=True).iterrows():
        essay_id, text = row["id"], row["text"]

        # Robust word/span alignment: regex over original text guarantees
        # words[i] always corresponds to word_spans[i] (no text.find drift).
        word_spans = [(m.start(), m.end()) for m in re.finditer(r"\S+", text)]
        words = [text[s:e] for s, e in word_spans]

        enc = tokenizer(text, max_length=CFG.max_length, padding="max_length",
                        truncation=True, return_offsets_mapping=True)
        offsets = enc["offset_mapping"]
        pred_labels = [id2label[p] for p in all_probs[idx].argmax(-1)]

        # token -> word
        word_labels = ["O"] * len(words)
        seen = set()
        for (ts, te), lab in zip(offsets, pred_labels):
            if ts == 0 and te == 0:
                continue
            for i, (ws, we) in enumerate(word_spans):
                if ws <= ts < we:
                    if i not in seen:
                        word_labels[i] = lab
                        seen.add(i)
                    break

        # Group consecutive same-class words into spans
        i = 0
        while i < len(word_labels):
            lab = word_labels[i]
            if lab == "O":
                i += 1; continue
            d_type = lab.split("-", 1)[1]
            j = i + 1
            while j < len(word_labels) and word_labels[j] != "O" and word_labels[j].split("-", 1)[1] == d_type:
                j += 1
            span = list(range(i, j))
            if len(span) >= MIN_WORDS.get(d_type, 3):
                submissions.append({
                    "id": essay_id,
                    "class": d_type,
                    "predictionstring": " ".join(map(str, span)),
                })
            i = j

    return pd.DataFrame(submissions)

# ---- Competition F1 metric (word-overlap, ≥50% from both sides) ----
def calc_overlap(pred_set, gt_set):
    inter = len(pred_set & gt_set)
    if inter == 0:
        return 0.0, 0.0
    return inter / len(pred_set), inter / len(gt_set)

def score_feedback_comp(pred_df, gt_df):
    """Macro F1 across discourse types, matching competition rules."""
    f1s = []
    for cls in DISCOURSE_TYPES:
        p = pred_df[pred_df["class"] == cls].copy().reset_index(drop=True)
        g = gt_df[gt_df["discourse_type"] == cls].copy().reset_index(drop=True)
        p["pred_id"] = p.index
        g["gt_id"] = g.index
        if len(p) == 0 and len(g) == 0:
            print(f"  {cls:22s} F1 = 0.0000  (no preds, no gt)")
            f1s.append(0.0); continue
        p["pred_set"] = p["predictionstring"].str.split().apply(set)
        g["gt_set"] = g["predictionstring"].str.split().apply(set)

        joined = p.merge(g, on="id", how="outer", suffixes=("_p", "_g"))

        TP = 0
        matched_gt, matched_pred = set(), set()
        for _, r in joined.iterrows():
            if pd.isna(r.get("pred_set")) or pd.isna(r.get("gt_set")):
                continue
            o1, o2 = calc_overlap(r["pred_set"], r["gt_set"])
            if o1 >= 0.5 and o2 >= 0.5 and r["pred_id"] not in matched_pred and r["gt_id"] not in matched_gt:
                TP += 1
                matched_pred.add(r["pred_id"]); matched_gt.add(r["gt_id"])
        FP = len(p) - TP
        FN = len(g) - TP
        f1 = TP / (TP + 0.5 * (FP + FN)) if (TP + FP + FN) > 0 else 0.0
        f1s.append(f1)
        print(f"  {cls:22s} F1 = {f1:.4f}")
    macro = float(np.mean(f1s))
    print(f"  {'MACRO':22s} F1 = {macro:.4f}")
    return macro

# ---- Run inference ----
test_df = pd.read_pickle("test_df.pkl")
ready_df = pd.read_pickle("ready_df.pkl")
train_data = pd.read_pickle("train_data.pkl")

tokenizer = LongformerTokenizerFast.from_pretrained(CFG.model_name, add_prefix_space=True)
model_paths = [f"{CFG.output_dir}/longformer_fold{f}.pt" for f in FOLDS_TO_USE]

# 1. Score on validation fold (sanity check + see real performance)
val_df = ready_df[ready_df["kfold"] == VAL_FOLD][["id", "text"]].reset_index(drop=True)
val_pred = predict(val_df, model_paths, tokenizer)
val_gt = train_data[train_data["id"].isin(val_df["id"])]
print("\n=== Validation scores ===")
score_feedback_comp(val_pred, val_gt)

# 2. Predict on test set
print("\n=== Test predictions ===")
sub = predict(test_df, model_paths, tokenizer)
sub.to_csv("submission.csv", index=False)

print(f"\nSubmission shape: {sub.shape}")
print(f"\nSample rows:")
print(sub.head(10).to_string())
print(f"\nClass distribution:")
print(sub["class"].value_counts())

Testing

In [ ]:
# ============================================================
# CELL 3: INFERENCE + SAMPLE RESULTS
# ============================================================
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from transformers import LongformerTokenizerFast
from tqdm import tqdm

# Tunable post-processing thresholds
MIN_WORDS = {"Lead": 9, "Position": 5, "Claim": 3, "Counterclaim": 6,
             "Rebuttal": 4, "Evidence": 14, "Concluding Statement": 11}

def predict(test_df, model_paths, tokenizer):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    test_ds = FeedbackDataset(test_df, tokenizer, CFG.max_length, train_data=None, is_train=False)
    test_loader = DataLoader(test_ds, batch_size=CFG.valid_batch_size, shuffle=False, num_workers=2)

    all_probs = None
    for path in model_paths:
        model = FeedbackModel(CFG.model_name, NUM_LABELS).to(device)
        model.load_state_dict(torch.load(path, map_location=device))
        model.eval()
        fold_probs = []
        with torch.no_grad():
            for batch in tqdm(test_loader, desc=f"Predict {path}"):
                with torch.cuda.amp.autocast():
                    out = model(
                        batch["input_ids"].to(device),
                        batch["attention_mask"].to(device),
                        batch["global_attention_mask"].to(device),
                    )
                fold_probs.append(torch.softmax(out["logits"], -1).cpu().numpy())
        fold_probs = np.concatenate(fold_probs, 0)
        all_probs = fold_probs if all_probs is None else all_probs + fold_probs
        del model; torch.cuda.empty_cache()
    all_probs /= len(model_paths)

    submissions = []
    for idx, row in test_df.reset_index(drop=True).iterrows():
        essay_id, text = row["id"], row["text"]
        words = text.split()

        enc = tokenizer(text, max_length=CFG.max_length, padding="max_length",
                        truncation=True, return_offsets_mapping=True)
        offsets = enc["offset_mapping"]
        pred_labels = [id2label[p] for p in all_probs[idx].argmax(-1)]

        # char position -> word index
        word_spans, cursor = [], 0
        for w in words:
            start = text.find(w, cursor)
            end = start + len(w)
            word_spans.append((start, end)); cursor = end

        # token -> word
        word_labels = ["O"] * len(words)
        seen = set()
        for (ts, te), lab in zip(offsets, pred_labels):
            if ts == 0 and te == 0:
                continue
            for i, (ws, we) in enumerate(word_spans):
                if ws <= ts < we:
                    if i not in seen:
                        word_labels[i] = lab
                        seen.add(i)
                    break

        # Group consecutive same-class words into spans
        i = 0
        while i < len(word_labels):
            lab = word_labels[i]
            if lab == "O":
                i += 1; continue
            d_type = lab.split("-", 1)[1]
            j = i + 1
            while j < len(word_labels) and word_labels[j] != "O" and word_labels[j].split("-", 1)[1] == d_type:
                j += 1
            span = list(range(i, j))
            if len(span) >= MIN_WORDS.get(d_type, 3):
                submissions.append({
                    "id": essay_id,
                    "class": d_type,
                    "predictionstring": " ".join(map(str, span)),
                })
            i = j

    return pd.DataFrame(submissions)

# ---- Competition F1 metric (word-overlap, ≥50% from both sides) ----
def calc_overlap(pred_set, gt_set):
    inter = len(pred_set & gt_set)
    if inter == 0:
        return 0.0, 0.0
    return inter / len(pred_set), inter / len(gt_set)

def score_feedback_comp(pred_df, gt_df):
    """Macro F1 across discourse types, matching competition rules."""
    f1s = []
    for cls in DISCOURSE_TYPES:
        p = pred_df[pred_df["class"] == cls].copy()
        g = gt_df[gt_df["discourse_type"] == cls].copy()
        p["pred_id"] = p.index
        g["gt_id"] = g.index
        p["pred_set"] = p["predictionstring"].str.split().apply(set)
        g["gt_set"] = g["predictionstring"].str.split().apply(set)

        joined = p.merge(g, left_on="id", right_on="id", how="outer", suffixes=("_p", "_g"))

        TP = 0
        matched_gt, matched_pred = set(), set()
        for _, r in joined.iterrows():
            if pd.isna(r.get("pred_set")) or pd.isna(r.get("gt_set")):
                continue
            o1, o2 = calc_overlap(r["pred_set"], r["gt_set"])
            if o1 >= 0.5 and o2 >= 0.5 and r["pred_id"] not in matched_pred and r["gt_id"] not in matched_gt:
                TP += 1
                matched_pred.add(r["pred_id"]); matched_gt.add(r["gt_id"])
        FP = len(p) - TP
        FN = len(g) - TP
        f1 = TP / (TP + 0.5 * (FP + FN)) if (TP + FP + FN) > 0 else 0.0
        f1s.append(f1)
        print(f"  {cls:22s} F1 = {f1:.4f}")
    macro = np.mean(f1s)
    print(f"  {'MACRO':22s} F1 = {macro:.4f}")
    return macro

# ---- Run inference ----
test_df = pd.read_pickle("test_df.pkl")
ready_df = pd.read_pickle("ready_df.pkl")
train_data = pd.read_pickle("train_data.pkl")

tokenizer = LongformerTokenizerFast.from_pretrained(CFG.model_name, add_prefix_space=True)
model_paths = [f"{CFG.output_dir}/longformer_fold{CFG.fold}.pt"]   # add more fold paths to ensemble

# 1. Score on validation fold (sanity check + see real performance)
val_df = ready_df[ready_df["kfold"] == CFG.fold][["id", "text"]].reset_index(drop=True)
val_pred = predict(val_df, model_paths, tokenizer)
val_gt = train_data[train_data["id"].isin(val_df["id"])]
print("\n=== Validation scores ===")
score_feedback_comp(val_pred, val_gt)

# 2. Predict on test set
print("\n=== Test predictions ===")
sub = predict(test_df, model_paths, tokenizer)
sub.to_csv("submission.csv", index=False)

print(f"\nSubmission shape: {sub.shape}")
print(f"\nSample rows:")
print(sub.head(10).to_string())
print(f"\nClass distribution:")
print(sub["class"].value_counts())

In [ ]:
# ============================================================
# SAVE MODEL ARTIFACTS — self-contained
# ============================================================
import shutil
import json
from pathlib import Path
from transformers import LongformerTokenizerFast

# Find whatever fold checkpoints exist in output dir
OUTPUT_DIR = Path("./output")
SAVE_DIR = Path("./saved_model")
SAVE_DIR.mkdir(exist_ok=True)

checkpoints = sorted(OUTPUT_DIR.glob("longformer_fold*.pt"))
print(f"Found {len(checkpoints)} checkpoint(s):")
for c in checkpoints:
    print(f"  {c.name} ({c.stat().st_size / (1024**2):.1f} MB)")

# 1. Copy checkpoints
for src in checkpoints:
    shutil.copy(src, SAVE_DIR / src.name)

# 2. Save tokenizer
tokenizer = LongformerTokenizerFast.from_pretrained(
    "allenai/longformer-base-4096", add_prefix_space=True
)
tokenizer.save_pretrained(SAVE_DIR / "tokenizer")

# 3. Save label maps + config so you can reload without redefining
DISCOURSE_TYPES = ["Lead", "Position", "Claim", "Counterclaim", "Rebuttal", "Evidence", "Concluding Statement"]
LABELS = ["O"] + [f"{p}-{t}" for t in DISCOURSE_TYPES for p in ["B", "I"]]
MIN_WORDS = {"Lead": 9, "Position": 5, "Claim": 3, "Counterclaim": 6,
             "Rebuttal": 4, "Evidence": 14, "Concluding Statement": 11}

artifacts = {
    "model_name": "allenai/longformer-base-4096",
    "max_length": 1024,
    "labels": LABELS,
    "label2id": {l: i for i, l in enumerate(LABELS)},
    "id2label": {str(i): l for i, l in enumerate(LABELS)},
    "discourse_types": DISCOURSE_TYPES,
    "min_words": MIN_WORDS,
    "checkpoints": [c.name for c in checkpoints],
}
with open(SAVE_DIR / "artifacts.json", "w") as f:
    json.dump(artifacts, f, indent=2)

# 4. Zip for download
shutil.make_archive("saved_model", "zip", SAVE_DIR)

print(f"\nSaved to {SAVE_DIR}/")
for p in sorted(SAVE_DIR.rglob("*")):
    if p.is_file():
        size_mb = p.stat().st_size / (1024**2)
        print(f"  {p.relative_to(SAVE_DIR)}: {size_mb:.1f} MB")
print(f"\nZipped: saved_model.zip ({Path('saved_model.zip').stat().st_size / (1024**2):.1f} MB)")

In [ ]:
from google.colab import files

# This will trigger your browser's download manager
files.download('saved_model.zip')

Reimport Zip

In [ ]:
# ============================================================
# CELL: Re-upload and unzip the saved model
# ============================================================
from google.colab import files
import zipfile, os

# 1. Upload the zip (browser dialog opens)
uploaded = files.upload()  # select saved_model.zip

# 2. Unzip it
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall(".")
    print("Extracted files:")
    for n in z.namelist():
        print(" ", n)

# 3. Sanity check that the .pt file is where we expect
# (adjust path below if the print above shows a different location)
!ls -la output/

KeyboardInterrupt: 